# Visualização inicial

Este notebook explora a base histórica de propostas normativas. Execute as células na ordem apresentada. Ele não altera o CSV de origem.

## 1. Preparar bibliotecas e localizar a base

O caminho é calculado a partir da pasta deste notebook; portanto, funciona mesmo quando o JupyterLab é aberto em outra pasta.

Importa as bibliotecas de análise e visualização, configura o estilo dos gráficos e localiza o arquivo CSV da base histórica.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 160)

NOME_ARQUIVO = "propostas_normativas_20210916_a_20260916.csv"
PASTAS_CANDIDATAS = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PASTA_POC = next((pasta for pasta in PASTAS_CANDIDATAS if (pasta / "data" / NOME_ARQUIVO).exists()), None)
if PASTA_POC is None:
    raise FileNotFoundError(f"Não encontrei data/{NOME_ARQUIVO}. Abra o JupyterLab dentro da pasta do projeto.")
ARQUIVO_DADOS = PASTA_POC / "data" / NOME_ARQUIVO
ARQUIVO_DADOS

## 2. Carregar os dados

`data_apresentacao` é convertida para data; `mes` será usado nos agrupamentos mensais.

Carrega o CSV, converte a data de apresentação, cria a coluna de mês e exibe o tamanho e o período coberto pela base.

In [ ]:
propostas = pd.read_csv(ARQUIVO_DADOS, parse_dates=["dataApresentacaoDocumento"])
propostas = propostas.rename(columns={"dataApresentacaoDocumento": "data_apresentacao"})
propostas["mes"] = propostas["data_apresentacao"].dt.to_period("M").dt.to_timestamp()

print(f"Proposições: {len(propostas):,}")
print(f"Período: {propostas['data_apresentacao'].min():%d/%m/%Y} a {propostas['data_apresentacao'].max():%d/%m/%Y}")
propostas.head(5)

Quantidade total de linhas da base carregada.

In [ ]:
print(f"Total de linhas do dataset: {len(propostas):,}")

## 3. Verificar as colunas e a qualidade mínima

Resume os tipos das colunas, a quantidade de valores ausentes e o número de valores distintos para uma verificação inicial da qualidade dos dados.

In [ ]:
resumo_qualidade = pd.DataFrame({
    "tipo": propostas.dtypes.astype(str),
    "ausentes": propostas.isna().sum(),
    "valores_unicos": propostas.nunique(),
})
resumo_qualidade

## 4. Distribuição por tipo documental

Conta as proposições por tipo documental e apresenta os resultados em uma tabela e em um gráfico de barras horizontais.

In [ ]:
por_tipo = propostas["tipoDocumento"].value_counts().rename_axis("tipo_documento").reset_index(name="quantidade")
display(por_tipo)

ax = por_tipo.sort_values("quantidade").plot.barh(
    x="tipo_documento", y="quantidade", legend=False, figsize=(10, 5), color="#0b6e4f"
)
ax.set(title="Propostas normativas por tipo documental", xlabel="Quantidade", ylabel="")
plt.tight_layout()
plt.show()

## 5. Evolução mensal total

Cada ponto representa o número de proposições apresentadas naquele mês.

Agrupa as proposições por mês, mostra os últimos meses disponíveis e desenha a evolução mensal do volume total.

In [ ]:
volume_mensal = propostas.groupby("mes").size().rename("quantidade")
display(volume_mensal.tail(12).to_frame())

ax = volume_mensal.plot(figsize=(12, 4), marker="o", color="#1f77b4")
ax.set(title="Volume mensal de propostas normativas", xlabel="Mês de apresentação", ylabel="Quantidade")
plt.tight_layout()
plt.show()

## 6. Evolução mensal por tipo

Esta é a primeira visão que será transformada em séries de treinamento na próxima etapa da PoC.

Agrupa as proposições por mês e tipo documental, reorganiza o resultado em uma série temporal e plota a evolução de cada tipo.

In [ ]:
serie_por_tipo = (
    propostas.groupby(["mes", "tipoDocumento"]).size()
    .rename("quantidade").reset_index()
    .pivot(index="mes", columns="tipoDocumento", values="quantidade")
    .fillna(0).astype(int)
)
display(serie_por_tipo.tail(12))

ax = serie_por_tipo.plot(figsize=(13, 6), marker="o")
ax.set(title="Volume mensal por tipo documental", xlabel="Mês de apresentação", ylabel="Quantidade")
ax.legend(title="Tipo documental", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Próximo passo

Depois de reconhecer o comportamento das séries, o próximo notebook pode criar defasagens temporais e comparar previsões simples (último valor, média móvel e mesmo mês do ano anterior) antes de usar modelos de aprendizado de máquina.